# Graphing the Data
## Imports

In [ ]:
import pandas as pd
import numpy as np
from currency_converter import CurrencyConverter, ECB_URL

# Needed for Plotly
import plotly.express as px
import plotly.tools as tools
import plotly.graph_objects as go

# Needed for Seaborn
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

## Run helpers and read in data

In [ ]:
%run helpers.ipynb

In [ ]:
oscar_with_imdb, imdb_non_oscar = read_data('oscar_with_imdb.csv', 'imdb_non_oscar.csv')

To plot all films at once regardless of nomination status, we use the columns as series to concatenate them into a data frame. This allows the budget and box office variables to be used as combined axes for Plotly Express.

In [ ]:
oscar_with_imdb['Status'] = oscar_with_imdb['Winner'].apply(lambda row: 'Winner' if row else 'Nominated')
imdb_non_oscar['Status'] = 'No nomination'

budget_nom = oscar_with_imdb['budget']
gross_nom = oscar_with_imdb['gross_worldwide']

budget_imdb = imdb_non_oscar['budget']
gross_imdb = imdb_non_oscar['gross_worldwide']

oscar_noms = pd.concat([budget_nom, gross_nom, oscar_with_imdb['Status']], axis=1)
non_noms = pd.concat([budget_imdb, gross_imdb, imdb_non_oscar['Status']], axis=1)

combined_films = pd.concat([oscar_noms, non_noms], axis=0)
combined_films.dropna(how='any', inplace=True)

In [ ]:
combined_films.shape

## Setting up data for scatter plot (Budget vs Box Office)

We want to categorize films by their academy award status or if they were not nominated. We begin by adding False entries to the `Winner` field in the joined data frame for Academy Award films, and then dropping any NA values in both joined data frames.

### Create `Status` field

In [ ]:
temp_oscar_with_imdb = oscar_with_imdb.dropna(subset=['budget', 'gross_worldwide'])
temp_imdb_non_oscar = imdb_non_oscar.dropna(subset=['budget', 'gross_worldwide'])

# Nominess are marked as NaN so we replace with False
temp_oscar_with_imdb['Winner'] = oscar_with_imdb['Winner'].fillna(False)

In [ ]:
temp_oscar_with_imdb['Status'] = temp_oscar_with_imdb['Winner'].apply(lambda row: 'Winner' if row else 'Nominated')

winners = temp_oscar_with_imdb[temp_oscar_with_imdb['Status'] == 'Winner']
nominees = temp_oscar_with_imdb[temp_oscar_with_imdb['Status'] == 'Nominated']
non_nom = temp_imdb_non_oscar

all_films = [winners, nominees, non_nom]
to_graph = [None, None, None]
film_groups = ['Won', 'Nominated', 'Not Nominated']

### Round budget and box and budget

Round everything and add to graph

In [ ]:
print([film.shape for film in all_films])

In [ ]:
## Initialize currency converter to pass into lambda

In [ ]:
currency_conv = CurrencyConverter('eurofxref-hist.csv', fallback_on_missing_rate=True, fallback_on_wrong_date=True)

In [ ]:
%run helpers.ipynb

In [ ]:
for i, df in enumerate(all_films):
    # ToDo check round_value function
    for j, row in df.iterrows():
        release_date = row['release_date']
    df['normalized_budget'] = df['budget'].apply(lambda x: normalize_budget(x, release_date, currency_conv))
        
    # df['adjusted_budget'] = df.apply(normalize_budget, axis=1)
    
    # df['rounded_budget'] = df['rounded_budget']
    # df['rounded_gross'] = np.log10(df['rounded_gross'])
    # to_graph[i] = df.groupby(['rounded_budget', 'rounded_gross']).size().reset_index(name='count')
    to_graph[i] = df

In [ ]:
[(1321, 31), (3993, 31), (8758, 20)]

In [ ]:
for i, df in enumerate(to_graph):
    df = df[df['normalized_budget'] != 0]
    to_graph[i] = df

In [ ]:
df1, df2, df3 = to_graph

In [ ]:
all_films_scatter = pd.concat([df1, df2, df3], ignore_index=True)

In [ ]:
all_films_scatter.shape

The `CurrencyConverter` library allowed for quick parsing, but had some failed lookups for currency. We have to drop films with currencies that did not properly convert or hard code the conversion rate. 

In [ ]:
for df in to_graph:
    df['rounded_budget'] = df['normalized_budget'].copy().apply(round_value)
    df['rounded_gross'] = df['gross_worldwide'].apply(round_value)
    
    # Drop NA and zero values
    df = df.dropna(subset=['rounded_budget', 'rounded_gross'])
    df = df[(df['rounded_budget'] > 0) & (df['rounded_gross'] > 0)]

In [ ]:
print([film.shape for film in to_graph])

### Iterate through rows and add points to scatter plot

In [ ]:
all_films_scatter['budget']

In [ ]:
# x = combined_films['budget']
# y = combined_films['gross_worldwide']
df1 = to_graph[0]
df2 = to_graph[1]
df3 = to_graph[2]
df3 = df3.rename(columns={'title': 'Film'})

all_films_scatter = pd.concat([df1, df2, df3], ignore_index=True)
fig = px.scatter(all_films_scatter, x='rounded_budget', y='rounded_gross', color='Status',
                 labels={'rounded_budget': 'Budget', 'rounded_gross': 'Gross', 'rating': 'IMDb Rating'},
                 hover_data={
                     'rating': True,
                     'Film': True if 'Film' in all_films_scatter.columns else False
                }
        )
# fig = px.scatter(x, y, log_x=True, log_y=True)
fig.update_layout(xaxis=dict(type='log'), xaxis_title='Budget (Log scaled)', yaxis_title='Worldwide Gross', title='Worldwide Box Office Earnings vs Film Budget')
# fig.update_layout(yaxis=dict(type='log'))
# fig.update_layout(yaxis=dict(type='log'))
fig.update_layout(height=600)
fig.show(config={'displayModeBar': False})

In [ ]:
# ToDo: Add scatter for revenue and color code profitability

In [ ]:
all_films_scatter['Profit'] = all_films_scatter['gross_worldwide'] - all_films_scatter['rounded_budget']

In [ ]:
# x = combined_films['budget']
# y = combined_films['gross_worldwide']
# df1 = to_graph[0]
# df2 = to_graph[1]
# df3 = to_graph[2]
# df3 = df3.rename(columns={'title': 'Film'})

# all_films_scatter = pd.concat([df1, df2, df3], ignore_index=True)
fig = px.scatter(all_films_scatter, x='rounded_budget', y='Profit', color='Status',
                hover_data={
                    'Film': True if 'Film' in all_films_scatter.columns else False
                }
        )
# fig = px.scatter(x, y, log_x=True, log_y=True)
# fig.update_layout(xaxis=dict(type='log'), xaxis_title='Budget (Log scaled)', yaxis_title='Worldwide Gross', title='Worldwide Box Office Earnings vs Film Budget')
# fig.update_layout(yaxis=dict(type='log'))
# fig.update_layout(yaxis=dict(type='log'))
fig.update_layout(height=600)
fig.show(config={'displayModeBar': False})

In [ ]:
profit_fig = px.scatter(
    all_films_scatter,
    x='budget',
    y='rounded_gross',
    size='votes',
    color='Status',
    hover_data=['Film'],
    labels={'rounded_budget': 'Budget', 'Profit': 'Profit', 'rounded_gross': 'Worldwide Gross'},
    title="Bubble Chart: Profit vs Budget vs Gross"
)
profit_fig.update_layout(height=700)
profit_fig.update_layout(xaxis=dict(type='log'), xaxis_title='Budget (Log scaled)', yaxis_title='Worldwide Gross', title='Worldwide Box Office Earnings vs Film Budget')
profit_fig.show()